# Teks — Katalog algoritma klasifikasi (satu contoh kode per algoritma)
**Kapan dipakai:** ingin melihat cara pakai tiap algoritma teks (ML klasik → RNN/CNN →
Transformer → Embedding) dengan data, split, dan metrik yang SAMA, supaya skornya bisa
dibandingkan langsung. Ini katalog, bukan jalur lomba: untuk lomba pakai `baseline_oof`
(TF-IDF) atau `finetune_oof` (transformer) yang sudah punya OOF + validator.

| Bagian | Algoritma |
|---|---|
| A. ML klasik | Naive Bayes, Logistic Regression, SVM, KNN, Decision Tree, Random Forest, |
| | XGBoost, LightGBM, CatBoost |
| B. Deep learning dari nol | RNN, LSTM, GRU, Bi-LSTM, Bi-GRU, Attention, Text-CNN, Transformer |
| C. Transformer pretrained | BERT/IndoBERT, DistilBERT, RoBERTa, DeBERTa, ALBERT, XLNet, ERNIE, |
| | T5, BART (semua lewat satu fungsi fine-tune) |
| D. Embedding kata | Word2Vec, GloVe, FastText (dilatih dari nol di data sendiri) |

Skor semua bagian dikumpulkan di akhir: `ringkasan_skor.csv` + grafik.
T5/BART untuk *membuat* teks (ringkasan) ada di `generasi`; di sini jadi pengklasifikasi.

**Data demo:** SmSA (sentimen Bahasa Indonesia), dipotong `MAX_ROWS` baris supaya cepat.

In [ ]:
import os
import re
import time
import zlib
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.tree import DecisionTreeClassifier
from torch import nn
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from xgboost import XGBClassifier

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"   # UBAH saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"
TEXT_COL = "text"      # UBAH
TARGET = "label"       # UBAH
MAX_ROWS = 3000        # UBAH: None = semua baris (lebih lama, terutama bagian C)
SEED = 42
MAX_LEN = 64           # kata per teks untuk bagian B
TOKENS = 128           # token per teks untuk bagian C
PRETRAINED = "../../../pretrained"
OUTPUT_FOLDER = "../../../outputs/text/katalog_algoritma"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)
print("device:", DEVICE)

## Baca data, split train / validasi
Label diubah ke angka 0..K-1 (XGBoost dan PyTorch butuh ini).
Semua bagian memakai split yang sama.

In [ ]:
data = pd.read_csv(TRAIN_CSV)
data[TEXT_COL] = data[TEXT_COL].fillna("").astype(str)
data[TEXT_COL] = data[TEXT_COL].str.lower()
if MAX_ROWS is not None and len(data) > MAX_ROWS:
    data = data.sample(MAX_ROWS, random_state=SEED)

encoder = LabelEncoder()
labels = encoder.fit_transform(data[TARGET])
N_CLASSES = len(encoder.classes_)
texts_train, texts_val, y_train, y_val = train_test_split(
    data[TEXT_COL].tolist(), labels, test_size=0.2, stratify=labels, random_state=SEED
)
print("train:", len(texts_train), "| val:", len(texts_val), "| kelas:", list(encoder.classes_))

scores = []   # satu baris per algoritma


def record(section, name, predictions, start):
    f1 = f1_score(y_val, predictions, average="macro")
    seconds = time.time() - start
    scores.append({"bagian": section, "algoritma": name, "f1_macro": round(f1, 4),
                   "detik": round(seconds, 1)})
    print(f"{section} | {name:<26} F1 macro {f1:.4f} | {seconds:.1f} s")


record("contoh", "tebak kelas terbanyak", np.full(len(y_val), np.bincount(y_train).argmax()),
       time.time())

## A. Machine Learning klasik (TF-IDF → model)
Model linear dan KNN memakai TF-IDF kata 1–2 gram. Model pohon (Decision Tree, Random Forest,
boosting) diberi kosakata lebih kecil (`max_features`) supaya tidak lambat.
- **Naive Bayes**: hitung peluang kata per kelas. Complement NB lebih tahan kelas timpang.
- **SVM**: `LinearSVC` (cepat, kuat untuk teks); `SVC(kernel="rbf")` = SVM non-linear, lambat.
- **KNN** dengan jarak cosine: kelas mayoritas dari teks paling mirip.

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(texts_train)
X_val_tfidf = tfidf.transform(texts_val)

linear_models = {
    "Naive Bayes (Multinomial)": MultinomialNB(alpha=0.1),
    "Naive Bayes (Complement)": ComplementNB(alpha=0.3),
    "Logistic Regression": LogisticRegression(C=5, max_iter=2000),
    "SVM (LinearSVC)": LinearSVC(C=0.5),
    "SVM (kernel RBF)": SVC(C=5, kernel="rbf"),
    "KNN (cosine)": KNeighborsClassifier(n_neighbors=15, metric="cosine", weights="distance"),
}
for name, model in linear_models.items():
    start = time.time()
    model.fit(X_train_tfidf, y_train)
    record("A ML", name, model.predict(X_val_tfidf), start)

In [ ]:
tfidf_small = TfidfVectorizer(min_df=2, max_features=3000, sublinear_tf=True)
X_train_small = tfidf_small.fit_transform(texts_train)
X_val_small = tfidf_small.transform(texts_val)

tree_models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=30, min_samples_leaf=2, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=6,
                             tree_method="hist", random_state=SEED),
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1,
                               random_state=SEED),
    # XGBoost di GPU: XGBClassifier(..., device="cuda")
}
for name, model in tree_models.items():
    start = time.time()
    model.fit(X_train_small, y_train)
    record("A ML", name, model.predict(X_val_small), start)

**CatBoost** bisa membaca kolom teks mentah sendiri (`text_features`): ia membuat fitur kata
(bag-of-words + Naive Bayes per kelas) di dalam. Tidak perlu TF-IDF. Kolom angka/kategori lain
bisa ditaruh di DataFrame yang sama (`cat_features=[...]`).

In [ ]:
start = time.time()
train_frame = pd.DataFrame()
train_frame["teks"] = texts_train
val_frame = pd.DataFrame()
val_frame["teks"] = texts_val
catboost = CatBoostClassifier(iterations=500, text_features=["teks"], verbose=0,
                              random_seed=SEED)
# GPU: CatBoostClassifier(..., task_type="GPU")
catboost.fit(train_frame, y_train)
record("A ML", "CatBoost (text_features)", catboost.predict(val_frame).ravel(), start)

## B. Deep learning dari nol (PyTorch)
Kata → id (kosakata dari train, id 0 = padding, id 1 = kata tak dikenal), dipotong/diisi
sampai `MAX_LEN` kata. Embedding dilatih dari awal (tanpa pretrained).

In [ ]:
counter = Counter()
for text in texts_train:
    counter.update(re.findall(r"\w+", text))
vocab = {"<pad>": 0, "<unk>": 1}
for word, count in counter.most_common(20000):
    if count >= 2:
        vocab[word] = len(vocab)


def word_ids(text):
    ids = []
    for word in re.findall(r"\w+", text):
        ids.append(vocab.get(word, 1))
    return ids


def encode(texts):
    ids = np.zeros((len(texts), MAX_LEN), dtype=np.int64)
    for row, text in enumerate(texts):
        words = word_ids(text)[:MAX_LEN]
        ids[row, :len(words)] = words
    return torch.tensor(ids)


ids_train = encode(texts_train)
ids_val = encode(texts_val)
print("kosakata:", len(vocab), "| bentuk:", tuple(ids_train.shape))

**RNN / LSTM / GRU / Bi-** dalam satu kelas. `kind` memilih sel: RNN biasa (mudah lupa kata awal,
*vanishing gradient*), LSTM (punya gerbang memori, konteks panjang), GRU (gerbang lebih sedikit,
lebih cepat). `bidirectional=True` membaca kiri→kanan dan kanan→kiri.
**Attention**: tiap kata diberi bobot penting (dipelajari), rangkuman = rata-rata berbobot;
tanpa attention rangkuman = rata-rata biasa semua kata.

In [ ]:
class RecurrentClassifier(nn.Module):
    def __init__(self, kind, bidirectional, attention, hidden=128):
        super().__init__()
        layers = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}
        self.embedding = nn.Embedding(len(vocab), 128, padding_idx=0)
        self.rnn = layers[kind](128, hidden, batch_first=True, bidirectional=bidirectional)
        out_size = hidden
        if bidirectional:
            out_size = hidden * 2
        self.attention = None
        if attention:
            self.attention = nn.Linear(out_size, 1)
        self.head = nn.Linear(out_size, N_CLASSES)

    def forward(self, ids):
        mask = (ids != 0).unsqueeze(-1).float()
        states, _ = self.rnn(self.embedding(ids))
        if self.attention is None:
            pooled = (states * mask).sum(1) / mask.sum(1).clamp(min=1)
        else:
            raw = self.attention(states)
            weights = torch.softmax(raw.masked_fill(mask == 0, -1e9), dim=1)
            pooled = (states * weights).sum(1)
        return self.head(pooled)


print(RecurrentClassifier("gru", True, True)(ids_train[:2]).shape)

**Text-CNN**: filter konvolusi 2/3/4 kata menangkap frasa pendek, lalu max-pool. Cepat, tapi
tidak paham urutan jauh. **Transformer dari nol**: embedding kata + posisi → self-attention
(`nn.TransformerEncoder`). Tanpa pretrained biasanya kalah dari model pretrained di bagian C.

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, sizes=(2, 3, 4), filters=100):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), 128, padding_idx=0)
        self.convs = nn.ModuleList()
        for size in sizes:
            self.convs.append(nn.Conv1d(128, filters, size, padding=size - 1))
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Linear(filters * len(sizes), N_CLASSES)

    def forward(self, ids):
        embedded = self.embedding(ids).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            pooled.append(torch.relu(conv(embedded)).amax(dim=2))
        return self.head(self.dropout(torch.cat(pooled, dim=1)))


print(TextCNN()(ids_train[:2]).shape)

In [ ]:
class SmallTransformer(nn.Module):
    def __init__(self, dim=128, heads=4, layers=2):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), dim, padding_idx=0)
        self.position = nn.Embedding(MAX_LEN, dim)
        layer = nn.TransformerEncoderLayer(dim, heads, dim * 2, dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Linear(dim, N_CLASSES)

    def forward(self, ids):
        padding = ids == 0
        padding[:, 0] = False   # teks kosong: jangan sampai semua posisi ditutup (hasil NaN)
        positions = torch.arange(ids.shape[1], device=ids.device)
        states = self.encoder(self.embedding(ids) + self.position(positions),
                              src_key_padding_mask=padding)
        mask = (~padding).unsqueeze(-1).float()
        return self.head((states * mask).sum(1) / mask.sum(1))


print(SmallTransformer()(ids_train[:2]).shape)

Satu fungsi latih untuk semua model bagian B (AdamW, cross-entropy, potong gradien).

In [ ]:
def predict_torch(model, ids):
    model.eval()
    outputs = []
    with torch.no_grad():
        for begin in range(0, len(ids), 256):
            outputs.append(model(ids[begin:begin + 256].to(DEVICE)).argmax(1).cpu())
    return torch.cat(outputs).numpy()


def train_torch(model, epochs=6, lr=2e-3, batch=64):
    torch.manual_seed(SEED)
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    targets = torch.tensor(y_train)
    for _ in range(epochs):
        model.train()
        order = torch.randperm(len(ids_train))
        for begin in range(0, len(order), batch):
            index = order[begin:begin + batch]
            logits = model(ids_train[index].to(DEVICE))
            loss = nn.functional.cross_entropy(logits, targets[index].to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
    return predict_torch(model, ids_val)


print("cek predict (model belum dilatih):", predict_torch(TextCNN().to(DEVICE), ids_val[:5]))

In [ ]:
recurrent_variants = {
    "RNN": ("rnn", False, False),
    "LSTM": ("lstm", False, False),
    "GRU": ("gru", False, False),
    "Bi-LSTM": ("lstm", True, False),
    "Bi-GRU": ("gru", True, False),
    "Bi-GRU + Attention": ("gru", True, True),
}
for name, (kind, bidirectional, attention) in recurrent_variants.items():
    start = time.time()
    torch.manual_seed(SEED)
    model = RecurrentClassifier(kind, bidirectional, attention)
    record("B DL", name, train_torch(model), start)

start = time.time()
torch.manual_seed(SEED)
record("B DL", "Text-CNN", train_torch(TextCNN()), start)
start = time.time()
torch.manual_seed(SEED)
record("B DL", "Transformer (dari nol)", train_torch(SmallTransformer(), lr=5e-4), start)

## C. Transformer pretrained (fine-tune)
Satu fungsi untuk semua model; yang berganti hanya folder di `MODEL_PATHS` (hapus `#` untuk
menyalakan). Folder yang tidak ada dilewati. `ignore_mismatched_sizes=True` membuang kepala
klasifikasi bawaan (mis. DeBERTa NLI 3 kelas) dan membuat yang baru sesuai jumlah kelas kita.
Semua folder di bawah diunduh oleh `download_model.bat` (`_alat/unduh_model.py`).

In [ ]:
MODEL_PATHS = {
    "IndoBERT (BERT)": f"{PRETRAINED}/indobert",
    "DistilBERT": f"{PRETRAINED}/distilbert_id",
    # "RoBERTa (XLM-R)": f"{PRETRAINED}/xlmr_base",
    # "DeBERTa (mDeBERTa)": f"{PRETRAINED}/nli_mdeberta",
    # "NusaBERT": f"{PRETRAINED}/nusabert",
    # "IndoBERTweet": f"{PRETRAINED}/indobertweet",
    # "IndoBERT large": f"{PRETRAINED}/indobert_large",   # 8 GB VRAM: BATCH 4
    # model Inggris (XLNet, ERNIE, BART) biasanya kalah di data Indonesia:
    # "ALBERT": f"{PRETRAINED}/albert_id",   # indobenchmark/indobert-lite-base-p1
    # "XLNet": f"{PRETRAINED}/xlnet_en",     # xlnet/xlnet-base-cased (Inggris)
    # "ERNIE": f"{PRETRAINED}/ernie_en",     # nghuyong/ernie-2.0-base-en (Inggris)
    # "T5 (IndoNanoT5)": f"{PRETRAINED}/indot5_base",   # LazarusNLP/IndoNanoT5-base
    # "BART": f"{PRETRAINED}/bart_en",       # facebook/bart-base (Inggris)
}
EPOCHS = 2
BATCH = 8      # kecil: laptop pernah mati saat beban GPU berat
LR = 3e-5

In [ ]:
def tokenize(tokenizer, texts):
    batch = tokenizer(list(texts), truncation=True, max_length=TOKENS, padding=True,
                      return_tensors="pt")
    return batch.to(DEVICE)


def predict_transformer(model, tokenizer):
    model.eval()
    outputs = []
    with torch.no_grad():
        for begin in range(0, len(texts_val), 32):
            batch = tokenize(tokenizer, texts_val[begin:begin + 32])
            outputs.append(model(**batch).logits.argmax(1).cpu())
    return torch.cat(outputs).numpy()


print("model yang akan dilatih:", list(MODEL_PATHS))

In [ ]:
def finetune(path):
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(
        path, num_labels=N_CLASSES, ignore_mismatched_sizes=True).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    texts = np.array(texts_train, dtype=object)
    targets = torch.tensor(y_train)
    for _ in range(EPOCHS):
        model.train()
        order = torch.randperm(len(texts))
        for begin in range(0, len(order), BATCH):
            index = order[begin:begin + BATCH]
            batch = tokenize(tokenizer, texts[index.numpy()])
            loss = model(**batch, labels=targets[index].to(DEVICE)).loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return predict_transformer(model, tokenizer)


for name, path in MODEL_PATHS.items():
    if not os.path.isdir(path):
        print("lewati, folder tidak ada:", path)
        continue
    start = time.time()
    torch.manual_seed(SEED)
    record("C Transformer", name, finetune(path), start)
    torch.cuda.empty_cache()

## D. Embedding kata dilatih dari nol
Tidak ada vektor kata pretrained offline, jadi ketiganya dilatih di teks train sendiri
(data lomba yang besar = vektor lebih bagus). Word2Vec dan GloVe menghasilkan vektor per kata;
vektor teks = rata-rata vektor katanya → Logistic Regression.
- **Word2Vec (skip-gram + negative sampling)**: kata tengah menebak kata di sekitarnya.
- **GloVe**: memfaktorkan tabel "berapa sering kata i muncul dekat kata j".
- **FastText**: kata + potongan huruf 3–5 → kata baru / salah ketik tetap dapat vektor.

In [ ]:
def doc_vectors(texts, vectors):
    out = np.zeros((len(texts), vectors.shape[1]), dtype=np.float32)
    for row, text in enumerate(texts):
        ids = word_ids(text)
        if ids:
            out[row] = vectors[ids].mean(0)
    return out


def score_vectors(name, vectors, start):
    classifier = make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=3000))
    classifier.fit(doc_vectors(texts_train, vectors), y_train)
    record("D Embedding", name, classifier.predict(doc_vectors(texts_val, vectors)), start)


def context_pairs(window=3):
    centers = []
    contexts = []
    for text in texts_train:
        ids = word_ids(text)
        for pos, center in enumerate(ids):
            for other in range(max(0, pos - window), min(len(ids), pos + window + 1)):
                if other != pos:
                    centers.append(center)
                    contexts.append(ids[other])
    return torch.tensor(centers), torch.tensor(contexts)


print("pasangan skip-gram:", len(context_pairs()[0]))

In [ ]:
class SkipGram(nn.Module):
    def __init__(self, dim=100):
        super().__init__()
        self.center = nn.Embedding(len(vocab), dim)
        self.context = nn.Embedding(len(vocab), dim)
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)
        nn.init.zeros_(self.context.weight)

    def forward(self, centers, contexts, negatives):
        center = self.center(centers)
        positive = (center * self.context(contexts)).sum(1)
        negative = torch.bmm(self.context(negatives), center.unsqueeze(2)).squeeze(2)
        positive_loss = nn.functional.logsigmoid(positive)
        negative_loss = nn.functional.logsigmoid(-negative).sum(1)
        return -(positive_loss + negative_loss).mean()

In [ ]:
def train_word2vec(epochs=5, batch=1024, negatives=5):
    centers, contexts = context_pairs()
    noise = torch.bincount(centers, minlength=len(vocab)).float() ** 0.75
    model = SkipGram().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    for _ in range(epochs):
        order = torch.randperm(len(centers))
        for begin in range(0, len(order), batch):
            index = order[begin:begin + batch]
            sampled = torch.multinomial(noise, len(index) * negatives, replacement=True)
            loss = model(centers[index].to(DEVICE), contexts[index].to(DEVICE),
                         sampled.view(len(index), negatives).to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    vectors = model.center.weight.detach()
    return vectors.cpu().numpy()


start = time.time()
torch.manual_seed(SEED)
score_vectors("Word2Vec + LogReg", train_word2vec(), start)

In [ ]:
def cooccurrence(window=5):
    counts = Counter()
    for text in texts_train:
        ids = word_ids(text)
        for pos, center in enumerate(ids):
            for other in range(pos + 1, min(len(ids), pos + window + 1)):
                counts[(center, ids[other])] += 1.0 / (other - pos)
                counts[(ids[other], center)] += 1.0 / (other - pos)
    pairs = torch.tensor(list(counts.keys()))
    values = torch.tensor(list(counts.values()), dtype=torch.float32)
    return pairs[:, 0], pairs[:, 1], values


print("pasangan ko-okurensi:", len(cooccurrence()[2]))

In [ ]:
class GloVe(nn.Module):
    def __init__(self, dim=100):
        super().__init__()
        self.word = nn.Embedding(len(vocab), dim)
        self.context = nn.Embedding(len(vocab), dim)
        self.word_bias = nn.Embedding(len(vocab), 1)
        self.context_bias = nn.Embedding(len(vocab), 1)
        for table in (self.word, self.context, self.word_bias, self.context_bias):
            nn.init.uniform_(table.weight, -0.05, 0.05)

    def forward(self, rows, cols, values):
        # rugi GloVe: f(X) * (w_i . w~_j + b_i + b~_j - log X)^2; f meredam pasangan sering
        dot = (self.word(rows) * self.context(cols)).sum(1)
        bias = self.word_bias(rows).squeeze(1) + self.context_bias(cols).squeeze(1)
        weight = torch.clamp(values / 100, max=1) ** 0.75
        return (weight * (dot + bias - torch.log(values)) ** 2).mean()

In [ ]:
def train_glove(epochs=30, batch=4096):
    rows, cols, values = cooccurrence()
    model = GloVe().to(DEVICE)
    optimizer = torch.optim.Adagrad(model.parameters(), lr=0.05)
    for _ in range(epochs):
        order = torch.randperm(len(values))
        for begin in range(0, len(order), batch):
            index = order[begin:begin + batch]
            loss = model(rows[index].to(DEVICE), cols[index].to(DEVICE),
                         values[index].to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    vectors = (model.word.weight + model.context.weight).detach()
    return vectors.cpu().numpy()


start = time.time()
torch.manual_seed(SEED)
score_vectors("GloVe + LogReg", train_glove(), start)

**FastText** (versi pengklasifikasi, seperti `fasttext supervised`): id kata + id potongan huruf
3–5 (di-hash ke `BUCKETS` slot) → rata-rata embedding (`nn.EmbeddingBag`) → linear.
Hash pakai `zlib.crc32` supaya id sama di tiap jalan (fungsi `hash()` Python berubah tiap sesi).

In [ ]:
BUCKETS = 200000


def ngram_ids(text):
    ids = [0]   # id 0 selalu ada supaya teks kosong tetap punya isi
    for word in re.findall(r"\w+", text):
        ids.append(vocab.get(word, 1))
        padded = f"<{word}>"
        for size in range(3, 6):
            for begin in range(len(padded) - size + 1):
                gram = padded[begin:begin + size].encode("utf-8")
                ids.append(len(vocab) + zlib.crc32(gram) % BUCKETS)
    return ids


def bag_batch(texts):
    flat = []
    offsets = []
    for text in texts:
        offsets.append(len(flat))
        flat.extend(ngram_ids(text))
    return torch.tensor(flat).to(DEVICE), torch.tensor(offsets).to(DEVICE)


print(ngram_ids("enak")[:8])

In [ ]:
class FastText(nn.Module):
    def __init__(self, dim=100):
        super().__init__()
        self.bag = nn.EmbeddingBag(len(vocab) + BUCKETS, dim, mode="mean")
        self.head = nn.Linear(dim, N_CLASSES)

    def forward(self, flat_ids, offsets):
        return self.head(self.bag(flat_ids, offsets))

In [ ]:
def train_fasttext(epochs=10, batch=32):
    model = FastText().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    texts = np.array(texts_train, dtype=object)
    targets = torch.tensor(y_train)
    for _ in range(epochs):
        order = torch.randperm(len(texts))
        for begin in range(0, len(order), batch):
            index = order[begin:begin + batch]
            flat, offsets = bag_batch(texts[index.numpy()])
            logits = model(flat, offsets)
            loss = nn.functional.cross_entropy(logits, targets[index].to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    with torch.no_grad():
        logits = model(*bag_batch(texts_val))
    return logits.argmax(1).cpu()


start = time.time()
torch.manual_seed(SEED)
record("D Embedding", "FastText (supervised)", train_fasttext().numpy(), start)

## Ringkasan skor
Satu split validasi saja: selisih kecil (< 0,01) belum tentu nyata. Untuk memilih model lomba,
bawa 2–3 teratas ke `baseline_oof` / `finetune_oof` (5 fold) lalu gabungkan di `ensemble_oof`.

In [ ]:
summary = pd.DataFrame(scores).sort_values("f1_macro", ascending=False)
summary.to_csv(f"{OUTPUT_FOLDER}/ringkasan_skor.csv", index=False)
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 0.32 * len(summary) + 1))
ax.barh(summary["algoritma"], summary["f1_macro"], color="#4C72B0")
ax.invert_yaxis()
ax.set_xlabel("F1 macro (validasi)")
for pos, value in enumerate(summary["f1_macro"]):
    ax.text(value, pos, f" {value:.3f}", va="center")
ax.grid(axis="x", linestyle="--", alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(f"{OUTPUT_FOLDER}/ringkasan_skor.png", dpi=120)
plt.show()